---
title: 'Phase 1 validation: testing a basic model'
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.5
  kernelspec:
    display_name: Python 3 (ipykernel)
    language: python
    name: python3
---


The goal of this notebook is to show that a classical-quantum model generative model can actually learn, using a simple model. For this, we would be using the torch library to create the classical part of our network, and add the quantum part through the functions and objects we created in `data.py` and `models.py`. You can find a more detailed explanation in the `docs`folder, including a schema of our model. Here, we will focus on explaining what we are doing instead of why.


In [1]:
# Add the src folder to the path, regardless of where the kernel's cwd is
import sys, pathlib

for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (parent / "src" / "data.py").exists():
        sys.path.insert(0, str(parent / "src"))
        break
else:
    raise RuntimeError("Could not locate the src/ folder")


In [10]:
from torch.nn import ReLU, Tanh

from data import * 
from models import *

import torch
import numpy as np

# We build the model
model = nn.Sequential(
    nn.Linear(6, 6),
    nn.Tanh(),
    QuantumCircuit(),
    nn.Linear(56, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)


Now, before training the model, we have to generate the desired output so we can train using the MDD loss.


In [11]:
desired_output = two_gaussian(256)
print(desired_output[:5])


tensor([[-1.3378, -0.3457],
        [-1.0752, -0.1302],
        [-0.7454,  0.2076],
        [-1.0948, -0.6346],
        [-0.9033, -0.3790]])


Let's also view this in a graph to ensure that this is working.


In [12]:
import matplotlib.pyplot as plt

points = desired_output.numpy() # tensor -> numpy array
plt.scatter(points[:, 0], points[:, 1])
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()


As we can see, we have two gaussians centered around (-1, 0) and (1, 0).

Now let's test the random_input_numbers function.


In [13]:
random_numers_test = []
for i in range(256): # Here, we make a loop instead of just calling random_input_numbers once, because of the seed. 
    # It is best to test it like a real simulation
    random_num_test = random_input_numbers(1, seed=i) # Also, we set the seed as i because taking the same seed
    # would just output 256 times the same numbers.
    random_numers_test.append(random_num_test.numpy())

all_values = np.array(random_numers_test).flatten()
plt.hist(all_values, bins=30)
plt.xlabel("value")
plt.ylabel("count")
plt.show()


As we can see here, we do have a random normal distribution, so the random_input_numbers function is properly working.

Now, before doing any training, let's see what result we get from running our model once.


In [14]:
outputs_test = []
for i in range(256):
    x = random_input_numbers(1, seed=i)
    y = model(x)
    outputs_test.append(y)

print(outputs_test[:5])


[tensor([[0.1428, 0.0825]], grad_fn=<AddmmBackward0>), tensor([[0.1437, 0.0811]], grad_fn=<AddmmBackward0>), tensor([[0.1386, 0.0792]], grad_fn=<AddmmBackward0>), tensor([[0.1418, 0.0821]], grad_fn=<AddmmBackward0>), tensor([[0.1455, 0.0776]], grad_fn=<AddmmBackward0>)]


In [15]:
xs = [t[0, 0].item() for t in outputs_test]
ys = [t[0, 1].item() for t in outputs_test]

plt.figure(figsize=(6, 6))
plt.scatter(xs, ys, alpha=0.6, s=15)
plt.xlabel("Output dim 0")
plt.ylabel("Output dim 1")
plt.title("Model outputs for 256 random inputs")
plt.grid(True, alpha=0.3)
plt.show()


As we can see, this is what our untrained model outputs. It looks nothing like the two gaussians we want. 


Let's just optimize a little bit computation time before that, by making batches of 256 at once.


In [18]:
x_opti = random_input_numbers(256, seed=42)
y_opti = model(x_opti)

print(y_opti[:5])


tensor([[0.1420, 0.0787],
        [0.1420, 0.0822],
        [0.1423, 0.0777],
        [0.1422, 0.0781],
        [0.1437, 0.0795],
        [0.1430, 0.0819],
        [0.1437, 0.0825],
        [0.1475, 0.0842],
        [0.1450, 0.0812],
        [0.1448, 0.0820],
        [0.1404, 0.0773],
        [0.1443, 0.0808],
        [0.1407, 0.0797],
        [0.1487, 0.0804],
        [0.1440, 0.0774],
        [0.1413, 0.0809],
        [0.1433, 0.0838],
        [0.1420, 0.0783],
        [0.1438, 0.0823],
        [0.1428, 0.0838],
        [0.1420, 0.0763],
        [0.1407, 0.0836],
        [0.1421, 0.0760],
        [0.1458, 0.0811],
        [0.1435, 0.0808],
        [0.1434, 0.0816],
        [0.1511, 0.0829],
        [0.1425, 0.0820],
        [0.1408, 0.0831],
        [0.1417, 0.0839],
        [0.1436, 0.0806],
        [0.1436, 0.0836],
        [0.1447, 0.0815],
        [0.1430, 0.0816],
        [0.1416, 0.0840],
        [0.1423, 0.0808],
        [0.1417, 0.0811],
        [0.1416, 0.0790],
        [0.1